In [ ]:
with open("data.txt", "r", encoding="utf-8") as f:
    text = f.read()

In [2]:
chars = sorted(list(set(text)))
vocab_size = len(chars)

In [3]:
stoi, itos = {}, {}
for i, char in enumerate(chars):
    stoi[char] = i
    itos[i] = char

encode = lambda s: [stoi[c] for c in s]
decode = lambda l: "".join([itos[i] for i in l])

In [4]:
import torch

data = torch.tensor(encode(text), dtype=torch.long)
data.shape, data.dtype

(torch.Size([1115390]), torch.int64)

In [5]:
data[:100]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59])

In [6]:
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

In [7]:
block_size = 8 # aka context length

train_data[:block_size + 1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [ ]:
x = train_data[:block_size]
y = train_data[1: block_size+1]
for t in range(block_size):
   context = x[:t + 1]
   target = y[t]
   print(f"{context} -> {target}")

tensor([18]) -> 47
tensor([18, 47]) -> 56
tensor([18, 47, 56]) -> 57
tensor([18, 47, 56, 57]) -> 58
tensor([18, 47, 56, 57, 58]) -> 1
tensor([18, 47, 56, 57, 58,  1]) -> 15
tensor([18, 47, 56, 57, 58,  1, 15]) -> 47
tensor([18, 47, 56, 57, 58,  1, 15, 47]) -> 58


In [9]:
torch.manual_seed(1337)
batch_size = 4 # how many independent sequences will be processed in parallel
block_size = 8 # max context length for predictions


def get_batch(split):
    data = train_data if split == "train" else val_data
    idx = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i: i + block_size] for i in idx])
    y = torch.stack([data[i + 1: i + block_size + 1] for i in idx])
    return x, y

In [10]:
xb, yb = get_batch("train")

In [11]:
xb

tensor([[43, 51,  1, 44, 39, 47, 56, 12],
        [46, 53, 53, 42,  1, 46, 43,  1],
        [52, 48, 59, 56, 47, 53, 59, 57],
        [47, 52, 45,  1, 47, 56, 53, 52]])

In [12]:
yb

tensor([[51,  1, 44, 39, 47, 56, 12,  0],
        [53, 53, 42,  1, 46, 43,  1, 57],
        [48, 59, 56, 47, 53, 59, 57,  1],
        [52, 45,  1, 47, 56, 53, 52,  8]])

In [13]:
for b in range(batch_size):
    for t in range(block_size):
        context = xb[b, :t + 1]
        target = yb[b, t]

        print(f"context: {context}; target {target}")

context: tensor([43]); target 51
context: tensor([43, 51]); target 1
context: tensor([43, 51,  1]); target 44
context: tensor([43, 51,  1, 44]); target 39
context: tensor([43, 51,  1, 44, 39]); target 47
context: tensor([43, 51,  1, 44, 39, 47]); target 56
context: tensor([43, 51,  1, 44, 39, 47, 56]); target 12
context: tensor([43, 51,  1, 44, 39, 47, 56, 12]); target 0
context: tensor([46]); target 53
context: tensor([46, 53]); target 53
context: tensor([46, 53, 53]); target 42
context: tensor([46, 53, 53, 42]); target 1
context: tensor([46, 53, 53, 42,  1]); target 46
context: tensor([46, 53, 53, 42,  1, 46]); target 43
context: tensor([46, 53, 53, 42,  1, 46, 43]); target 1
context: tensor([46, 53, 53, 42,  1, 46, 43,  1]); target 57
context: tensor([52]); target 48
context: tensor([52, 48]); target 59
context: tensor([52, 48, 59]); target 56
context: tensor([52, 48, 59, 56]); target 47
context: tensor([52, 48, 59, 56, 47]); target 53
context: tensor([52, 48, 59, 56, 47, 53]); targ

In [ ]:
from IPython.terminal.pt_inputhooks.osx import C
from encodings.punycode import T
from sympy.physics.secondquant import B
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLM(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        # idx and targets = (B, seq_length)
        logits = self.token_embedding_table(idx) # (B, seq_len, vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B * T, C)
            targets = targets.view(B * T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx = (B, seq_len)
        for _ in range(max_new_tokens):
            logits, _ = self(idx)
            # logits = (B, seq_len, vocab_size)
            # get the last time step
            logits = logits[:, -1, :] # (B, vocab_size)
            probs = F.softmax(logits, dim=-1) # (B, vocab_size)
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            idx = torch.cat((idx, idx_next), dim=1) # (B, seq_len + 1)

        return idx

In [77]:
m = BigramLM(vocab_size)
logits, loss = m(xb, yb)
loss

tensor(4.7894, grad_fn=<NllLossBackward0>)

In [78]:
print(decode(m.generate(torch.zeros((1, 1), dtype=torch.long), max_new_tokens=100)[0].tolist()))


&yuLFBFRtdxAqmu
Ypzx: r q.nqbwEJvA
&fWOEXdnrHvnb?T uNw&&LyuBhbIMRBYX-Hai:HRIOdtoe,fpNz.EGbwuV-S:js$f


In [79]:
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [86]:
batch_size = 32
for step in range(10000):
    xb, yb = get_batch("train")
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(loss.item())

2.4207756519317627


In [88]:
print(decode(m.generate(torch.zeros((1, 1), dtype=torch.long), max_new_tokens=300)[0].tolist()))



A d, he aler:
GRKI w menake nd:
ARTivalithoramyouth walon se avoif ENas brth my onithishing.

I By weant sn Lous ty,-
Wint wist at:
IIth 'BUCinldoonoud. wands! ms my sbo nhe, is peiofey winom ic MI a con st tlodof
g sisir cchee berfoownde s ad whow TRD thit oudsus sownomath arour m buret
Ane,
Tourd


In [89]:
torch.manual_seed(1337)
B, T, C = 4, 8, 2
x = torch.randn(B, T, C)
x.shape

torch.Size([4, 8, 2])

In [95]:
xbow = torch.zeros((B, T, C))
for b in range(B):
    for t in range(T):
        xprev = x[b, :t + 1]
        xbow[b, t] = xprev.mean(0)


In [ ]:
wei = torch.tril(torch.ones(T, T))
wei /= wei.sum(1, keepdim=True)
xbow2 = wei @ x

In [116]:
tril = torch.tril(torch.ones(T, T))
wei = torch.zeros(T, T)
wei = wei.masked_fill(tril == 0, float("-inf"))
wei = F.softmax(wei, dim=-1)
wei

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]])

In [97]:
torch.manual_seed(42)
a = torch.tril(torch.ones(3, 3))
a /= torch.sum(a, 1, keepdim=True)
b = torch.randint(0, 10, (3, 2)).float()
c = a @ b

In [125]:
torch.manual_seed(1337)
B, T, C = 4, 8, 32
x = torch.randn(B, T, C)

head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)

k = key(x) # (B, T, head_size)
q = query(x) # (B, T, head_size)
v = value(x) # (B, T, head_size)

wei = q @ k.transpose(-2, -1) # (B, T, head_size) @ (B, head_size, T) --> (B, T, T)
# for every row of B, we now have TxT matrix of token affinities

tril = torch.tril(torch.ones(T, T))
wei = wei.masked_fill(tril == 0, float("-inf"))
wei = F.softmax(wei, dim=-1)
out = wei @ v
out


tensor([[[-1.5713e-01,  8.8009e-01,  1.6152e-01, -7.8239e-01, -1.4289e-01,
           7.4676e-01,  1.0068e-01, -5.2395e-01, -8.8726e-01,  1.9067e-01,
           1.7616e-01, -5.9426e-01, -4.8124e-01, -4.8599e-01,  2.8623e-01,
           5.7099e-01],
         [ 6.7643e-01, -5.4770e-01, -2.4780e-01,  3.1430e-01, -1.2798e-01,
          -2.9521e-01, -4.2962e-01, -1.0891e-01, -4.9282e-02,  7.2679e-01,
           7.1296e-01, -1.1639e-01,  3.2665e-01,  3.4315e-01, -7.0975e-02,
           1.2716e+00],
         [ 4.8227e-01, -1.0688e-01, -4.0555e-01,  1.7696e-01,  1.5811e-01,
          -1.6967e-01,  1.6217e-02,  2.1509e-02, -2.4903e-01, -3.7725e-01,
           2.7867e-01,  1.6295e-01, -2.8951e-01, -6.7610e-02, -1.4162e-01,
           1.2194e+00],
         [ 1.9708e-01,  2.8561e-01, -1.3028e-01, -2.6552e-01,  6.6781e-02,
           1.9535e-01,  2.8074e-02, -2.4511e-01, -4.6466e-01,  6.9287e-02,
           1.5284e-01, -2.0324e-01, -2.4789e-01, -1.6213e-01,  1.9474e-01,
           7.6778e-01],
    

In [126]:
wei[0]

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1574, 0.8426, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2088, 0.1646, 0.6266, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5792, 0.1187, 0.1889, 0.1131, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0294, 0.1052, 0.0469, 0.0276, 0.7909, 0.0000, 0.0000, 0.0000],
        [0.0176, 0.2689, 0.0215, 0.0089, 0.6812, 0.0019, 0.0000, 0.0000],
        [0.1691, 0.4066, 0.0438, 0.0416, 0.1048, 0.2012, 0.0329, 0.0000],
        [0.0210, 0.0843, 0.0555, 0.2297, 0.0573, 0.0709, 0.2423, 0.2391]],
       grad_fn=<SelectBackward0>)

In [127]:
torch.softmax(torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5]), dim=-1)

tensor([0.1925, 0.1426, 0.2351, 0.1426, 0.2872])

In [ ]:
n_embed = 32

class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embed, head_size, bias=False)
        self.query = nn.Linear(n_embed, head_size, bias=False)
        self.value = nn.Linear(n_embed, head_size, bias=False)
        self.register_buffer(
            "tril", torch.tril(torch.ones(block_size, block_size))
        )  # tril is not a model param

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)  # (B, T, head_size)
        q = self.query(x)  # (B, T, head_size)
        v = self.value(x)  # (B, T, head_size)

        wei = (
            q @ k.transpose(-2, -1) * C**-0.5
        )  # (B, T, head_size) @ (B, head_size, T) --> (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))  # (B, T, T)
        wei = F.softmax(wei, dim=-1)

        out = wei @ v  # (B, T, T) @ (B, T, head_size) --> (B, T, head_size)
        return out


class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embed, n_embed)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.proj(out)
        return out

In [148]:
x = torch.randn(4, 8, 32)

mha = MultiHeadAttention(4, n_embed // 4)

In [149]:
mha(x)

torch.Size([4, 8, 32])
torch.Size([4, 8, 32])


tensor([[[ 0.5683,  0.1262,  0.4342,  ...,  0.0195,  0.4623,  0.3445],
         [ 0.3469,  0.1464,  0.5275,  ..., -0.0081,  0.5144,  0.5607],
         [ 0.1638, -0.0278,  0.3763,  ..., -0.0035,  0.2711,  0.3621],
         ...,
         [ 0.1728,  0.0272,  0.0578,  ...,  0.1428,  0.2696,  0.2780],
         [ 0.1704, -0.0205,  0.0146,  ...,  0.1661,  0.3093,  0.2491],
         [ 0.2169, -0.0274, -0.0768,  ...,  0.1552,  0.2518,  0.3041]],

        [[ 0.5569, -1.0292, -0.4427,  ...,  0.0575, -0.6162, -0.1727],
         [ 0.3575, -0.2753, -0.3035,  ..., -0.1138, -0.2391, -0.2600],
         [ 0.2141, -0.2172, -0.3703,  ..., -0.0236, -0.0841, -0.2143],
         ...,
         [ 0.1265, -0.2689, -0.3134,  ...,  0.0341, -0.0956, -0.1083],
         [ 0.1256, -0.2314, -0.2082,  ..., -0.0032, -0.0597, -0.1147],
         [ 0.1596, -0.2668, -0.2165,  ...,  0.0207, -0.0711, -0.0982]],

        [[ 0.1346,  0.4049, -0.0108,  ...,  0.3334,  0.0156,  0.6799],
         [-0.2085,  0.2717, -0.0900,  ...,  0